# Data Pre-processing: NASA POWER Data-Quality Validation

Validates the monthly NASA POWER dataset (GHI, cloud amount, relative humidity) for every grid cell before modelling.

**Input**
- `data/raw/nasa_power/nasa_power_monthly.csv`
- `data/processed/grid_metadata.csv`
- `data/raw/elevation/grid_elevation.csv`

**Output**
- `data/processed/quality/validation_summary.csv`: one row per check (PASS / WARN / FAIL)
- `data/processed/quality/flagged_records.csv`: every flagged value, with the reason
- `data/processed/nasa_power_validated.csv`: dataset with extraterrestrial irradiance, clearness index and per-variable quality flags

**Severity**
- **FAIL**: the dataset must be fixed before modelling (structure errors, missing values, physically impossible values).
- **WARN**: the value is physically possible but unusual; it is kept and flagged for review.

**Invalid vs. outlier**
- *Invalid* means physically impossible: negative GHI, GHI above the extraterrestrial (top-of-atmosphere) irradiance, cloud amount or RH outside 0–100 %, or a NASA fill value (-999). These are set to NaN in the validated dataset.
- *Implausible* means GHI that is possible but outside the realistic clearness-index range for monthly means.
- *Outlier* means statistically unusual for that grid cell and calendar month (robust z-score). Outliers are **kept**, since they can be real events such as typhoons or El Niño months.

In [1]:
"""
validate_nasa_power.py

Purpose:
    Data-quality validation of the monthly NASA POWER dataset
    before pre-processing and modelling.

Variables:
    ghi_kwh_m2_day = Global Horizontal Irradiance (kWh/m²/day)
    cloud_amt_pct  = Cloud Amount (%)
    rh2m_pct       = Relative Humidity at 2 Meters (%)

Input:
    data/raw/nasa_power/nasa_power_monthly.csv
    data/processed/grid_metadata.csv
    data/raw/elevation/grid_elevation.csv

Output:
    data/processed/quality/validation_summary.csv
    data/processed/quality/flagged_records.csv
    data/processed/nasa_power_validated.csv
"""

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 1. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

DATA_FILE = Path(
    "data/raw/nasa_power/nasa_power_monthly.csv"
)

GRID_FILE = Path(
    "data/processed/grid_metadata.csv"
)

ELEVATION_FILE = Path(
    "data/raw/elevation/grid_elevation.csv"
)


# ------------------------------------------------------------
# OUTPUT FILES
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "data/processed/quality"
)

SUMMARY_FILE = (
    OUTPUT_DIR /
    "validation_summary.csv"
)

FLAGS_FILE = (
    OUTPUT_DIR /
    "flagged_records.csv"
)

VALIDATED_FILE = Path(
    "data/processed/nasa_power_validated.csv"
)


# ------------------------------------------------------------
# STUDY PERIOD
# ------------------------------------------------------------

START_DATE = "2005-01-01"

END_DATE = "2024-12-01"

EXPECTED_MONTHS = 240


# ------------------------------------------------------------
# VARIABLES
# ------------------------------------------------------------

GHI_COL = "ghi_kwh_m2_day"

CLOUD_COL = "cloud_amt_pct"

RH_COL = "rh2m_pct"

VARIABLES = [
    GHI_COL,
    CLOUD_COL,
    RH_COL
]


# ------------------------------------------------------------
# PHYSICAL LIMITS
# ------------------------------------------------------------

# NASA POWER missing-data fill value
NASA_FILL_VALUE = -999

# Cloud amount and relative humidity are percentages
VALID_RANGES = {
    CLOUD_COL: (0.0, 100.0),
    RH_COL:    (0.0, 100.0)
}

# GHI lower limit is 0; the upper limit is the extraterrestrial
# (top-of-atmosphere) irradiation H0 computed per grid and month.
SOLAR_CONSTANT = 1361.0   # W/m² (Kopp & Lean, 2011)

# Realistic range of the monthly-mean clearness index Kt = GHI / H0.
# Values outside this range are possible but implausible for
# monthly means and are flagged as WARN.
KT_PLAUSIBLE_RANGE = (0.10, 0.80)


# ------------------------------------------------------------
# OUTLIER SETTINGS
# ------------------------------------------------------------

# Modified z-score threshold (Iglewicz & Hoaglin, 1993)
ROBUST_Z_THRESHOLD = 3.5

# Consecutive identical monthly values that count as a flat line
FLATLINE_RUN_LENGTH = 3


# ------------------------------------------------------------
# CONSISTENCY SETTINGS
# ------------------------------------------------------------

COORD_TOLERANCE = 1e-6


# ============================================================
# 2. CREATE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\nLoading data...")

for file in [DATA_FILE, GRID_FILE, ELEVATION_FILE]:

    if not file.exists():

        raise FileNotFoundError(
            f"Required file was not found: {file}"
        )


df = pd.read_csv(
    DATA_FILE,
    parse_dates=["date"]
)

grid = pd.read_csv(GRID_FILE)

elevation = pd.read_csv(ELEVATION_FILE)


print(f"NASA POWER records: {len(df)}")
print(f"Grid cells (metadata): {len(grid)}")
print(f"Grid cells (elevation): {len(elevation)}")


# ============================================================
# 4. RESULT COLLECTORS
# ============================================================

results = []

flag_tables = []


def add_result(section, check, n_issues, detail, severity="FAIL"):
    """
    Record the outcome of one validation check.
    A check with zero issues is PASS; otherwise it takes
    the given severity (FAIL or WARN).
    """

    status = "PASS" if n_issues == 0 else severity

    results.append({
        "section":  section,
        "check":    check,
        "status":   status,
        "n_issues": int(n_issues),
        "detail":   detail
    })

    print(f"[{status:<4}] {check}: {detail}")


def add_flags(rows, variable, flag, detail):
    """
    Store flagged records in long format:
    one row per flagged value.
    """

    if rows.empty:
        return

    table = rows[["grid_number", "grid_id", "date"]].copy()

    table["variable"] = variable

    table["value"] = rows[variable].values if variable in rows else np.nan

    table["flag"] = flag

    table["detail"] = detail

    flag_tables.append(table)


Loading data...
NASA POWER records: 22320
Grid cells (metadata): 93
Grid cells (elevation): 93


## A. Structural checks

In [2]:
# ============================================================
# A. STRUCTURAL CHECKS
# ============================================================

print("\n" + "=" * 65)
print("A. STRUCTURAL CHECKS")
print("=" * 65)

SECTION = "A. Structure"


# ------------------------------------------------------------
# A1. Required columns
# ------------------------------------------------------------

required_columns = [
    "grid_number",
    "grid_id",
    "latitude",
    "longitude",
    "year",
    "month",
    "date",
    *VARIABLES
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

add_result(
    SECTION,
    "Required columns present",
    len(missing_columns),
    f"missing: {missing_columns}" if missing_columns else "all required columns present"
)

if missing_columns:

    raise ValueError(
        f"Cannot continue validation. Missing columns: {missing_columns}"
    )


# ------------------------------------------------------------
# A2. Numeric data types
# ------------------------------------------------------------

non_numeric = [
    column
    for column in VARIABLES + ["latitude", "longitude", "year", "month"]
    if not pd.api.types.is_numeric_dtype(df[column])
]

add_result(
    SECTION,
    "Variables are numeric",
    len(non_numeric),
    f"non-numeric: {non_numeric}" if non_numeric else "all variables numeric"
)


# ------------------------------------------------------------
# A3. Grid coverage matches grid metadata
# ------------------------------------------------------------

data_grids = set(df["grid_number"])

meta_grids = set(grid["grid_number"])

missing_grids = sorted(meta_grids - data_grids)

extra_grids = sorted(data_grids - meta_grids)

add_result(
    SECTION,
    "Grid coverage matches grid_metadata.csv",
    len(missing_grids) + len(extra_grids),
    f"{len(data_grids)} grids in data, {len(meta_grids)} in metadata; "
    f"missing: {missing_grids}, extra: {extra_grids}"
)


# ------------------------------------------------------------
# A4. grid_id and coordinates match grid metadata
# ------------------------------------------------------------

per_grid = (
    df.groupby("grid_number")
    .agg(
        grid_id=("grid_id", "first"),
        n_grid_id=("grid_id", "nunique"),
        latitude=("latitude", "first"),
        n_latitude=("latitude", "nunique"),
        longitude=("longitude", "first"),
        n_longitude=("longitude", "nunique")
    )
    .reset_index()
    .merge(
        grid[["grid_number", "grid_id", "query_latitude", "query_longitude"]],
        on="grid_number",
        how="left",
        suffixes=("", "_meta")
    )
)

inconsistent = per_grid[
    (per_grid["n_grid_id"] > 1)
    | (per_grid["n_latitude"] > 1)
    | (per_grid["n_longitude"] > 1)
    | (per_grid["grid_id"] != per_grid["grid_id_meta"])
    | ((per_grid["latitude"] - per_grid["query_latitude"]).abs() > COORD_TOLERANCE)
    | ((per_grid["longitude"] - per_grid["query_longitude"]).abs() > COORD_TOLERANCE)
]

add_result(
    SECTION,
    "grid_id and coordinates consistent with metadata",
    len(inconsistent),
    f"inconsistent grids: {inconsistent['grid_number'].tolist()}"
    if len(inconsistent) else "all grids consistent"
)


# ------------------------------------------------------------
# A5. Every grid has exactly 240 months
# ------------------------------------------------------------

months_per_grid = df.groupby("grid_number").size()

wrong_count = months_per_grid[months_per_grid != EXPECTED_MONTHS]

add_result(
    SECTION,
    f"Every grid has exactly {EXPECTED_MONTHS} months",
    len(wrong_count),
    f"grids with wrong count: {wrong_count.to_dict()}"
    if len(wrong_count) else f"all {len(months_per_grid)} grids have {EXPECTED_MONTHS} months"
)


# ------------------------------------------------------------
# A6. No duplicate grid_number + date
# ------------------------------------------------------------

duplicate_mask = df.duplicated(
    subset=["grid_number", "date"],
    keep=False
)

add_result(
    SECTION,
    "No duplicate grid_number + date",
    duplicate_mask.sum(),
    f"{duplicate_mask.sum()} duplicated rows"
)

add_flags(
    df[duplicate_mask],
    "date",
    "duplicate",
    "duplicate grid_number + date"
)


# ------------------------------------------------------------
# A7. Continuous monthly sequence, Jan 2005 - Dec 2024
# ------------------------------------------------------------

expected_dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="MS"
)

gap_records = []

for grid_number, group in df.groupby("grid_number"):

    dates = pd.DatetimeIndex(group["date"].unique())

    missing_dates = expected_dates.difference(dates)

    extra_dates = dates.difference(expected_dates)

    if len(missing_dates) or len(extra_dates):

        gap_records.append({
            "grid_number":   grid_number,
            "missing_dates": [d.strftime("%Y-%m") for d in missing_dates],
            "extra_dates":   [d.strftime("%Y-%m") for d in extra_dates]
        })

add_result(
    SECTION,
    "Continuous monthly sequence 2005-01 to 2024-12",
    len(gap_records),
    f"grids with gaps or out-of-period dates: {gap_records}"
    if gap_records else
    f"all grids continuous from {expected_dates[0]:%Y-%m} to {expected_dates[-1]:%Y-%m}"
)


# ------------------------------------------------------------
# A8. Dates are first-of-month and match year / month columns
# ------------------------------------------------------------

date_mismatch = (
    (df["date"].dt.day != 1)
    | (df["date"].dt.year != df["year"])
    | (df["date"].dt.month != df["month"])
)

add_result(
    SECTION,
    "date matches year and month columns",
    date_mismatch.sum(),
    f"{date_mismatch.sum()} mismatched rows"
)

add_flags(
    df[date_mismatch],
    "date",
    "date_mismatch",
    "date does not match year/month or is not first of month"
)


# ------------------------------------------------------------
# A9. Records sorted by grid and date
# ------------------------------------------------------------

is_sorted = df[["grid_number", "date"]].equals(
    df[["grid_number", "date"]]
    .sort_values(["grid_number", "date"])
    .reset_index(drop=True)
)

add_result(
    SECTION,
    "Records sorted by grid_number and date",
    0 if is_sorted else 1,
    "sorted" if is_sorted else "not sorted (will be sorted in validated output)",
    severity="WARN"
)

df = df.sort_values(
    ["grid_number", "date"]
).reset_index(drop=True)


A. STRUCTURAL CHECKS
[PASS] Required columns present: all required columns present
[PASS] Variables are numeric: all variables numeric
[PASS] Grid coverage matches grid_metadata.csv: 93 grids in data, 93 in metadata; missing: [], extra: []
[PASS] grid_id and coordinates consistent with metadata: all grids consistent
[PASS] Every grid has exactly 240 months: all 93 grids have 240 months
[PASS] No duplicate grid_number + date: 0 duplicated rows
[PASS] Continuous monthly sequence 2005-01 to 2024-12: all grids continuous from 2005-01 to 2024-12
[PASS] date matches year and month columns: 0 mismatched rows
[PASS] Records sorted by grid_number and date: sorted


## B. Missing values

In [3]:
# ============================================================
# B. MISSING VALUES
# ============================================================

print("\n" + "=" * 65)
print("B. MISSING VALUES")
print("=" * 65)

SECTION = "B. Missing values"


for variable in VARIABLES:

    nan_mask = df[variable].isna()

    fill_mask = df[variable] == NASA_FILL_VALUE

    add_result(
        SECTION,
        f"No missing {variable}",
        nan_mask.sum() + fill_mask.sum(),
        f"{nan_mask.sum()} NaN, {fill_mask.sum()} NASA fill values ({NASA_FILL_VALUE})"
    )

    add_flags(df[nan_mask], variable, "missing", "NaN")

    add_flags(df[fill_mask], variable, "missing", f"NASA fill value {NASA_FILL_VALUE}")


# Treat NASA fill values as missing from here on
df[VARIABLES] = df[VARIABLES].replace(
    NASA_FILL_VALUE,
    np.nan
)


B. MISSING VALUES
[PASS] No missing ghi_kwh_m2_day: 0 NaN, 0 NASA fill values (-999)
[PASS] No missing cloud_amt_pct: 0 NaN, 0 NASA fill values (-999)
[PASS] No missing rh2m_pct: 0 NaN, 0 NASA fill values (-999)


## C. Physical validity

GHI is checked against the monthly-mean daily **extraterrestrial irradiation H0**, the solar energy reaching the top of the atmosphere at each grid's latitude. Surface GHI cannot exceed H0, so a clearness index `Kt = GHI / H0` above 1 is physically impossible.

H0 is computed for every day and averaged over each month (Duffie & Beckman, 2013, Eqs. 1.4.1, 1.6.1a, 1.10.3).

In [4]:
# ============================================================
# C. PHYSICAL VALIDITY
# ============================================================

print("\n" + "=" * 65)
print("C. PHYSICAL VALIDITY")
print("=" * 65)

SECTION = "C. Physical validity"


# ------------------------------------------------------------
# C1. Extraterrestrial irradiation H0 (kWh/m²/day)
# ------------------------------------------------------------

def daily_extraterrestrial_irradiation(latitude_deg, day_of_year):
    """
    Daily extraterrestrial irradiation on a horizontal surface
    in kWh/m²/day (Duffie & Beckman, 2013).
    """

    phi = np.radians(latitude_deg)

    # Eccentricity correction (Eq. 1.4.1)
    e0 = 1 + 0.033 * np.cos(2 * np.pi * day_of_year / 365)

    # Solar declination, Cooper (1969) (Eq. 1.6.1a)
    delta = np.radians(23.45) * np.sin(2 * np.pi * (284 + day_of_year) / 365)

    # Sunset hour angle
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))

    # Daily extraterrestrial irradiation (Eq. 1.10.3), Wh/m² -> kWh/m²
    return (
        (24 / np.pi) * SOLAR_CONSTANT * e0
        * (np.cos(phi) * np.cos(delta) * np.sin(ws) + ws * np.sin(phi) * np.sin(delta))
        / 1000
    )


days = pd.date_range(
    START_DATE,
    pd.Timestamp(END_DATE) + pd.offsets.MonthEnd(0),
    freq="D"
)

daily = (
    df[["grid_number", "latitude"]]
    .drop_duplicates()
    .merge(pd.DataFrame({"day": days}), how="cross")
)

daily["h0"] = daily_extraterrestrial_irradiation(
    daily["latitude"].to_numpy(),
    daily["day"].dt.dayofyear.to_numpy()
)

daily["date"] = daily["day"].dt.to_period("M").dt.to_timestamp()

monthly_h0 = (
    daily.groupby(["grid_number", "date"], as_index=False)["h0"]
    .mean()
    .rename(columns={"h0": "h0_kwh_m2_day"})
)

df = df.merge(
    monthly_h0,
    on=["grid_number", "date"],
    how="left"
)

df["kt"] = df[GHI_COL] / df["h0_kwh_m2_day"]

print(
    f"H0 range: {df['h0_kwh_m2_day'].min():.3f} - "
    f"{df['h0_kwh_m2_day'].max():.3f} kWh/m²/day"
)


# ------------------------------------------------------------
# C2. GHI invalid: negative or above extraterrestrial
# ------------------------------------------------------------

ghi_negative = df[GHI_COL] < 0

ghi_above_h0 = df[GHI_COL] > df["h0_kwh_m2_day"]

add_result(
    SECTION,
    "GHI >= 0 (invalid if negative)",
    ghi_negative.sum(),
    f"{ghi_negative.sum()} negative values"
)

add_result(
    SECTION,
    "GHI <= extraterrestrial H0 (invalid if Kt > 1)",
    ghi_above_h0.sum(),
    f"{ghi_above_h0.sum()} values above H0"
)

add_flags(df[ghi_negative], GHI_COL, "invalid", "negative GHI")

add_flags(df[ghi_above_h0], GHI_COL, "invalid", "GHI exceeds extraterrestrial H0 (Kt > 1)")


# ------------------------------------------------------------
# C3. GHI implausible: clearness index outside realistic range
# ------------------------------------------------------------

kt_low, kt_high = KT_PLAUSIBLE_RANGE

ghi_implausible = (
    ~(ghi_negative | ghi_above_h0)
    & ((df["kt"] < kt_low) | (df["kt"] > kt_high))
)

add_result(
    SECTION,
    f"Clearness index Kt within {kt_low}-{kt_high}",
    ghi_implausible.sum(),
    f"{ghi_implausible.sum()} values outside range; "
    f"observed Kt {df['kt'].min():.3f} - {df['kt'].max():.3f}",
    severity="WARN"
)

add_flags(
    df[ghi_implausible],
    GHI_COL,
    "implausible",
    f"Kt outside {kt_low}-{kt_high}"
)


# ------------------------------------------------------------
# C4. Cloud amount and RH within 0 - 100 %
# ------------------------------------------------------------

range_masks = {}

for variable, (low, high) in VALID_RANGES.items():

    mask = (df[variable] < low) | (df[variable] > high)

    range_masks[variable] = mask

    add_result(
        SECTION,
        f"{variable} within {low:g}-{high:g}",
        mask.sum(),
        f"{mask.sum()} values out of range; "
        f"observed {df[variable].min():.2f} - {df[variable].max():.2f}"
    )

    add_flags(df[mask], variable, "invalid", f"outside {low:g}-{high:g}")


invalid_masks = {
    GHI_COL:   ghi_negative | ghi_above_h0,
    CLOUD_COL: range_masks[CLOUD_COL],
    RH_COL:    range_masks[RH_COL]
}


C. PHYSICAL VALIDITY
H0 range: 6.944 - 10.993 kWh/m²/day
[PASS] GHI >= 0 (invalid if negative): 0 negative values
[PASS] GHI <= extraterrestrial H0 (invalid if Kt > 1): 0 values above H0
[PASS] Clearness index Kt within 0.1-0.8: 0 values outside range; observed Kt 0.276 - 0.718
[PASS] cloud_amt_pct within 0-100: 0 values out of range; observed 7.97 - 99.09
[PASS] rh2m_pct within 0-100: 0 values out of range; observed 56.02 - 94.30


## D. Statistical outliers

A value is an outlier when its **modified z-score** within the same grid cell and calendar month exceeds 3.5 (Iglewicz & Hoaglin, 1993):

`M = 0.6745 × (x − median) / MAD`

Comparing each value with the same calendar month, for example every January in grid 12, removes the seasonal cycle, so normal wet-season months are not flagged. Median and MAD are robust, so a single extreme year does not hide itself. Outliers are **flagged but kept**.

In [5]:
# ============================================================
# D. STATISTICAL OUTLIERS
# ============================================================

print("\n" + "=" * 65)
print("D. STATISTICAL OUTLIERS")
print("=" * 65)

SECTION = "D. Outliers"


def modified_z_score(values):
    """
    Modified z-score (Iglewicz & Hoaglin, 1993).
    Returns 0 when the MAD is zero.
    """

    median = values.median()

    mad = (values - median).abs().median()

    if pd.isna(mad) or mad == 0:
        return pd.Series(0.0, index=values.index)

    return 0.6745 * (values - median) / mad


outlier_masks = {}

for variable in VARIABLES:

    # Exclude invalid values so they do not distort median / MAD
    clean = df[variable].where(~invalid_masks[variable])

    z_column = f"{variable}_robust_z"

    df[z_column] = (
        clean.groupby([df["grid_number"], df["month"]])
        .transform(modified_z_score)
    )

    mask = df[z_column].abs() > ROBUST_Z_THRESHOLD

    outlier_masks[variable] = mask

    add_result(
        SECTION,
        f"{variable} outliers (|modified z| > {ROBUST_Z_THRESHOLD})",
        mask.sum(),
        f"{mask.sum()} outliers in {df.loc[mask, 'grid_number'].nunique()} grids "
        f"({mask.mean() * 100:.2f}% of records)",
        severity="WARN"
    )

    flagged = df[mask]

    add_flags(
        flagged,
        variable,
        "outlier",
        "modified z = " + flagged[z_column].round(2).astype(str)
    )


D. STATISTICAL OUTLIERS
[WARN] ghi_kwh_m2_day outliers (|modified z| > 3.5): 137 outliers in 69 grids (0.61% of records)
[WARN] cloud_amt_pct outliers (|modified z| > 3.5): 224 outliers in 91 grids (1.00% of records)
[WARN] rh2m_pct outliers (|modified z| > 3.5): 249 outliers in 79 grids (1.12% of records)


## E. Additional checks

- **Flat lines**: the same value repeated for 3 or more consecutive months usually indicates a gap-filled or stuck series.
- **GHI–cloud consistency**: cloud amount should be negatively correlated with GHI in every grid. A zero or positive correlation suggests a mismatch between variables or a coordinate error.
- **Elevation coverage**: every grid needs an elevation value for the Ineichen-Perez clear-sky model.

In [6]:
# ============================================================
# E. ADDITIONAL CHECKS
# ============================================================

print("\n" + "=" * 65)
print("E. ADDITIONAL CHECKS")
print("=" * 65)

SECTION = "E. Additional"


# ------------------------------------------------------------
# E1. Flat-line (stuck) values
# ------------------------------------------------------------

for variable in VARIABLES:

    same_as_previous = (
        df.groupby("grid_number")[variable]
        .diff()
        .eq(0)
    )

    run_id = (~same_as_previous).cumsum()

    run_length = df.groupby(run_id)[variable].transform("size")

    mask = run_length >= FLATLINE_RUN_LENGTH

    add_result(
        SECTION,
        f"{variable} has no flat lines (>= {FLATLINE_RUN_LENGTH} identical months)",
        mask.sum(),
        f"{mask.sum()} records in flat runs",
        severity="WARN"
    )

    add_flags(
        df[mask],
        variable,
        "flatline",
        f">= {FLATLINE_RUN_LENGTH} consecutive identical values"
    )


# ------------------------------------------------------------
# E2. GHI - cloud amount correlation per grid
# ------------------------------------------------------------

ghi_cloud_corr = (
    df.groupby("grid_number")[[GHI_COL, CLOUD_COL]]
    .corr()
    .xs(GHI_COL, level=1)[CLOUD_COL]
)

non_negative = ghi_cloud_corr[ghi_cloud_corr >= 0]

add_result(
    SECTION,
    "GHI negatively correlated with cloud amount in every grid",
    len(non_negative),
    f"r range {ghi_cloud_corr.min():.3f} to {ghi_cloud_corr.max():.3f}; "
    f"non-negative grids: {non_negative.index.tolist()}",
    severity="WARN"
)


# ------------------------------------------------------------
# E3. Elevation available for every grid
# ------------------------------------------------------------

elevation_grids = set(
    elevation.loc[elevation["elevation_m"].notna(), "grid_number"]
)

missing_elevation = sorted(data_grids - elevation_grids)

add_result(
    SECTION,
    "Elevation available for every grid",
    len(missing_elevation),
    f"grids without elevation: {missing_elevation}"
    if missing_elevation else f"all {len(data_grids)} grids have elevation"
)


E. ADDITIONAL CHECKS
[PASS] ghi_kwh_m2_day has no flat lines (>= 3 identical months): 0 records in flat runs
[PASS] cloud_amt_pct has no flat lines (>= 3 identical months): 0 records in flat runs
[PASS] rh2m_pct has no flat lines (>= 3 identical months): 0 records in flat runs
[PASS] GHI negatively correlated with cloud amount in every grid: r range -0.868 to -0.355; non-negative grids: []
[PASS] Elevation available for every grid: all 93 grids have elevation


## F. Summary and outputs

In [7]:
# ============================================================
# F. SUMMARY AND OUTPUTS
# ============================================================

# ------------------------------------------------------------
# F1. Validation summary
# ------------------------------------------------------------

summary = pd.DataFrame(results)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)


# ------------------------------------------------------------
# F2. Flagged records
# ------------------------------------------------------------

flag_columns = ["grid_number", "grid_id", "date", "variable", "value", "flag", "detail"]

flags = (
    pd.concat(flag_tables, ignore_index=True)
    if flag_tables else pd.DataFrame(columns=flag_columns)
)

flags.to_csv(
    FLAGS_FILE,
    index=False
)


# ------------------------------------------------------------
# F3. Validated dataset with per-variable quality flags
# ------------------------------------------------------------
# Flag priority: missing > invalid > implausible > outlier > valid
# Invalid values are set to NaN; outliers are kept.

validated = df.copy()

flag_names = {
    GHI_COL:   "ghi_flag",
    CLOUD_COL: "cloud_amt_flag",
    RH_COL:    "rh2m_flag"
}

for variable, flag_column in flag_names.items():

    implausible = ghi_implausible if variable == GHI_COL else pd.Series(False, index=df.index)

    validated[flag_column] = np.select(
        [
            df[variable].isna(),
            invalid_masks[variable],
            implausible,
            outlier_masks[variable]
        ],
        [
            "missing",
            "invalid",
            "implausible",
            "outlier"
        ],
        default="valid"
    )

    validated.loc[invalid_masks[variable], variable] = np.nan


validated = validated.drop(
    columns=[f"{variable}_robust_z" for variable in VARIABLES]
)

validated.to_csv(
    VALIDATED_FILE,
    index=False
)


# ------------------------------------------------------------
# F4. Report
# ------------------------------------------------------------

n_fail = (summary["status"] == "FAIL").sum()

n_warn = (summary["status"] == "WARN").sum()

n_pass = (summary["status"] == "PASS").sum()


print("\n")
print("=" * 65)
print("DATA-QUALITY VALIDATION SUMMARY")
print("=" * 65)

print(
    summary[["section", "check", "status", "n_issues"]]
    .to_string(index=False)
)

print("\nFlag counts per variable:")

print(
    validated[list(flag_names.values())]
    .apply(pd.Series.value_counts)
    .fillna(0)
    .astype(int)
    .to_string()
)

print(f"\nPASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")

if n_fail:
    print("\nRESULT: DATASET NOT READY. Resolve FAIL checks before modelling.")
elif n_warn:
    print("\nRESULT: DATASET USABLE. Review WARN checks in flagged_records.csv.")
else:
    print("\nRESULT: DATASET PASSED ALL CHECKS.")

print(f"\nSummary:   {SUMMARY_FILE}")
print(f"Flags:     {FLAGS_FILE}")
print(f"Validated: {VALIDATED_FILE}")

print("\nDone.")



DATA-QUALITY VALIDATION SUMMARY
             section                                                     check status  n_issues
        A. Structure                                  Required columns present   PASS         0
        A. Structure                                     Variables are numeric   PASS         0
        A. Structure                   Grid coverage matches grid_metadata.csv   PASS         0
        A. Structure          grid_id and coordinates consistent with metadata   PASS         0
        A. Structure                         Every grid has exactly 240 months   PASS         0
        A. Structure                           No duplicate grid_number + date   PASS         0
        A. Structure            Continuous monthly sequence 2005-01 to 2024-12   PASS         0
        A. Structure                       date matches year and month columns   PASS         0
        A. Structure                    Records sorted by grid_number and date   PASS         0
   B. 

## References

- Duffie, J. A., & Beckman, W. A. (2013). *Solar engineering of thermal processes* (4th ed.). Wiley. https://doi.org/10.1002/9781118671603
- Iglewicz, B., & Hoaglin, D. C. (1993). *How to detect and handle outliers*. ASQC Quality Press.
- Kopp, G., & Lean, J. L. (2011). A new, lower value of total solar irradiance: Evidence and climate significance. *Geophysical Research Letters, 38*(1), L01706. https://doi.org/10.1029/2010GL045777